In [ ]:
# -*- coding: utf-8 -*-
# WiDS Datathon 2023 — Limited Feature Set (13 features) with Optuna Tuning
# Target: contest-tmp2m-14d__tmp2m
# Metric: RMSE (lower is better), R² reported as well
# Models: XGBoost + LightGBM ensemble + SHAP analysis

import numpy as np
import pandas as pd
import warnings
import os
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('All imports OK')

# select features
SELECTED_FEATURES = [
    'nmme-tmp2m-56w__gfdlflora',
    'nmme-tmp2m-34w__cfsv2',
    'nmme-tmp2m-34w__gfdlflorb',
    'nmme-tmp2m-56w__cfsv2',
    'contest-pevpr-sfc-gauss-14d__pevpr',
    'nmme-tmp2m-34w__gfdlflora',
    'contest-prwtr-eatm-14d__prwtr',
    'contest-wind-h500-14d__wind-hgt-500',
    'contest-wind-h100-14d__wind-hgt-100',
    'lat',
    'contest-slp-14d__slp',
    'nmme-tmp2m-34w__nasa',
    'wind-vwnd-250-2010-9'
]
SELECTED_FEATURES = list(dict.fromkeys(SELECTED_FEATURES))
print(f'Selected features ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}')

TARGET = 'contest-tmp2m-14d__tmp2m'
SPLIT_DATE = pd.to_datetime('2016-04-08')

# reduce memory_usage
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type in numerics:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                for dtype in [np.int8, np.int16, np.int32, np.int64]:
                    if c_min > np.iinfo(dtype).min and c_max < np.iinfo(dtype).max:
                        df[col] = df[col].astype(dtype)
                        break
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f'Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB ({100*(start_mem-end_mem)/start_mem:.1f}% reduction)')
    return df

# load datasets
BASE = '/kaggle/input/datasets/ericrhadoophop/widsdata2023'

df_train = pd.read_csv(f'{BASE}/train_data.csv')
df_test  = pd.read_csv(f'{BASE}/test_data.csv')

print('--- Train ---')
df_train = reduce_mem_usage(df_train)
print('--- Test  ---')
df_test  = reduce_mem_usage(df_test)

print(f'\nTrain shape : {df_train.shape}')
print(f'Test  shape : {df_test.shape}')

# keep the key features
if 'startdate' not in df_train.columns:
    raise KeyError("Column 'startdate' not found in training data")
if 'index' not in df_test.columns:
    raise KeyError("Column 'index' not found in test data")

train_cols = SELECTED_FEATURES + [TARGET, 'startdate']
df_train = df_train[train_cols].copy()
test_cols = SELECTED_FEATURES + ['index']
df_test = df_test[test_cols].copy()

for col in SELECTED_FEATURES:
    if col not in df_test.columns:
        df_test[col] = np.nan
        print(f'Warning: {col} not in test set, filled with NaN.')

print(f'\nTrain shape after selection: {df_train.shape}')
print(f'Test shape after selection:  {df_test.shape}')

# time split
df_train['startdate'] = pd.to_datetime(df_train['startdate'], errors='coerce')
train_mask = df_train['startdate'] < SPLIT_DATE
val_mask   = df_train['startdate'] >= SPLIT_DATE

print(f'\nSplit date: {SPLIT_DATE.date()}')
print(f'Train samples: {train_mask.sum():,}')
print(f'Val samples:   {val_mask.sum():,}')
print(f'Val ratio: {val_mask.sum()/len(df_train):.2%}')

# split X,y val,train
X = df_train[SELECTED_FEATURES].values
y = df_train[TARGET].values

X_train = X[train_mask.values]
y_train = y[train_mask.values]
X_val   = X[val_mask.values]
y_val   = y[val_mask.values]

X_test_raw = df_test[SELECTED_FEATURES].values

print(f'\nX_train shape: {X_train.shape}')
print(f'X_val shape:   {X_val.shape}')
print(f'X_test shape:  {X_test_raw.shape}')

# ============================================================
# Missing value imputation   Standardization (both fitted on the training set)
# ============================================================
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_val_sc   = scaler.transform(X_val_imp)
X_test_sc  = scaler.transform(X_test_imp)

print('Imputation and scaling done (fit on train only).')

# ============================================================
# Optuna Hyperparameter Optimization
# ============================================================
N_TRIALS = 30  # Thirty times

# ---------- XGBoost Objective function ----------
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'early_stopping_rounds': 50,
        'eval_metric': 'rmse',
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0
    }
    model = XGBRegressor(**params)
    model.fit(X_train_sc, y_train, eval_set=[(X_val_sc, y_val)], verbose=False)
    pred = model.predict(X_val_sc)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    return rmse

print(f'\n=== XGBoost Optuna Optimization ({N_TRIALS} trials) ===')
xgb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

xgb_best_params = xgb_study.best_params
xgb_best_rmse = xgb_study.best_value

# Calculate the R² corresponding to the optimal parameters
best_xgb = XGBRegressor(**xgb_best_params)
best_xgb.fit(X_train_sc, y_train, eval_set=[(X_val_sc, y_val)], verbose=False)
xgb_best_r2 = r2_score(y_val, best_xgb.predict(X_val_sc))

print(f'\nXGBoost best Val RMSE : {xgb_best_rmse:.5f}')
print(f'XGBoost best Val R²   : {xgb_best_r2:.5f}')
print('XGBoost best params:')
for k, v in xgb_best_params.items():
    print(f'  {k:25s}: {v}')

# ---------- LightGBM Objective function ----------
def lgbm_objective(trial):
    params = {
        'n_estimators': 2000,  # Fix a larger value, early stopping determines the actual number of trees
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    model = LGBMRegressor(**params)
    model.fit(
        X_train_sc, y_train,
        eval_set=[(X_val_sc, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    pred = model.predict(X_val_sc)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    return rmse

print(f'\n=== LightGBM Optuna Optimization ({N_TRIALS} trials) ===')
lgbm_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=True)

lgbm_best_params = lgbm_study.best_params
lgbm_best_rmse = lgbm_study.best_value

# Calculate the R² corresponding to the optimal parameters
best_lgbm = LGBMRegressor(**lgbm_best_params)
best_lgbm.fit(
    X_train_sc, y_train,
    eval_set=[(X_val_sc, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)
lgbm_best_r2 = r2_score(y_val, best_lgbm.predict(X_val_sc))

print(f'\nLightGBM best Val RMSE : {lgbm_best_rmse:.5f}')
print(f'LightGBM best Val R²   : {lgbm_best_r2:.5f}')
print('LightGBM best params:')
for k, v in lgbm_best_params.items():
    print(f'  {k:25s}: {v}')

# ============================================================
# Train the final model using the best hyperparameters (full training set)
# ============================================================
# Combine training and validation sets for final training
X_all = np.vstack([X_train_sc, X_val_sc])
y_all = np.concatenate([y_train, y_val])

# XGBoost final model (with early stopping)
xgb_final_params = xgb_best_params.copy()
xgb_final_params['n_estimators'] = 2000
xgb_final_params['early_stopping_rounds'] = 50   
xgb_final_params['eval_metric'] = 'rmse'
xgb_final = XGBRegressor(**xgb_final_params)
xgb_final.fit(X_all, y_all, eval_set=[(X_val_sc, y_val)], verbose=False)
# After early stopping, the model automatically has best_iteration
xgb_best_n = xgb_final.best_iteration

# LightGBM final model
lgbm_final_params = lgbm_best_params.copy()
lgbm_final_params['n_estimators'] = 2000
lgbm_final = LGBMRegressor(**lgbm_final_params)
lgbm_final.fit(
    X_all, y_all,
    eval_set=[(X_val_sc, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)
lgbm_best_n = lgbm_final.best_iteration_

print(f'\nFinal XGBoost  best trees: {xgb_best_n}')
print(f'Final LightGBM best trees: {lgbm_best_n}')

# ============================================================
# Predictions and ensemble on the validation set
# ============================================================
xgb_val_pred = xgb_final.predict(X_val_sc)
lgbm_val_pred = lgbm_final.predict(X_val_sc)

xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_val_pred))
xgb_r2 = r2_score(y_val, xgb_val_pred)
lgbm_rmse = np.sqrt(mean_squared_error(y_val, lgbm_val_pred))
lgbm_r2 = r2_score(y_val, lgbm_val_pred)

total_err = xgb_rmse + lgbm_rmse
w_xgb = lgbm_rmse / total_err
w_lgbm = xgb_rmse / total_err

ensemble_val = w_xgb * xgb_val_pred + w_lgbm * lgbm_val_pred
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_val))
ensemble_r2 = r2_score(y_val, ensemble_val)

print('\n=== Validation Performance (Final Models) ===')
print(f'  XGBoost   : RMSE = {xgb_rmse:.5f}, R² = {xgb_r2:.5f}')
print(f'  LightGBM  : RMSE = {lgbm_rmse:.5f}, R² = {lgbm_r2:.5f}')
print(f'  Ensemble  : RMSE = {ensemble_rmse:.5f}, R² = {ensemble_r2:.5f}')

# ============================================================
# SHAP Model Interpretability Analysis
# ============================================================
try:
    import shap
    print("\n=== SHAP Analysis ===")
    
    # For quick computation, use a subset of the validation set (for example, 500 samples)
    shap_sample_size = min(500, len(X_val_sc))
    X_shap_sample = X_val_sc[:shap_sample_size]
    y_shap_sample = y_val[:shap_sample_size]
    
    # 1. XGBoost SHAP
    print("Computing SHAP values for XGBoost...")
    explainer_xgb = shap.TreeExplainer(xgb_final)
    shap_values_xgb = explainer_xgb.shap_values(X_shap_sample)
    
    # Feature importance (mean of absolute values)
    shap_mean_abs_xgb = np.mean(np.abs(shap_values_xgb), axis=0)
    feature_importance_xgb = pd.DataFrame({
        'feature': SELECTED_FEATURES,
        'importance': shap_mean_abs_xgb
    }).sort_values('importance', ascending=False)
    
    print("\nXGBoost Feature Importance (mean |SHAP|):")
    print(feature_importance_xgb.to_string(index=False))
    
    # Save SHAP summary plot
    shap.summary_plot(shap_values_xgb, X_shap_sample, feature_names=SELECTED_FEATURES, show=False)
    plt = shap.summary_plot(shap_values_xgb, X_shap_sample, feature_names=SELECTED_FEATURES, show=False)
    import matplotlib.pyplot as plt
    plt.tight_layout()
    plt.savefig('shap_summary_xgboost.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved SHAP summary plot: shap_summary_xgboost.png")
    
    # 2. LightGBM SHAP
    print("Computing SHAP values for LightGBM...")
    explainer_lgb = shap.TreeExplainer(lgbm_final)
    shap_values_lgb = explainer_lgb.shap_values(X_shap_sample)
    
    shap_mean_abs_lgb = np.mean(np.abs(shap_values_lgb), axis=0)
    feature_importance_lgb = pd.DataFrame({
        'feature': SELECTED_FEATURES,
        'importance': shap_mean_abs_lgb
    }).sort_values('importance', ascending=False)
    
    print("\nLightGBM Feature Importance (mean |SHAP|):")
    print(feature_importance_lgb.to_string(index=False))
    
    shap.summary_plot(shap_values_lgb, X_shap_sample, feature_names=SELECTED_FEATURES, show=False)
    plt = shap.summary_plot(shap_values_lgb, X_shap_sample, feature_names=SELECTED_FEATURES, show=False)
    plt.tight_layout()
    plt.savefig('shap_summary_lightgbm.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved SHAP summary plot: shap_summary_lightgbm.png")
    
except ImportError:
    print("\nSHAP library not installed. Skipping SHAP analysis.")
except Exception as e:
    print(f"\nSHAP analysis failed: {e}")

# ============================================================
# Generate submission file
# ============================================================
xgb_test_pred = xgb_final.predict(X_test_sc)
lgbm_test_pred = lgbm_final.predict(X_test_sc)
ensemble_test = w_xgb * xgb_test_pred + w_lgbm * lgbm_test_pred

submission = pd.DataFrame({
    'index': df_test['index'],
    'contest-tmp2m-14d__tmp2m': ensemble_test
})
submission.to_csv('submission.csv', index=False)
print(f'\nSubmission saved: submission.csv ({len(submission):,} rows)')
print(submission.head())
print(f'\nPrediction range: [{ensemble_test.min():.3f}, {ensemble_test.max():.3f}]')
print(f'Prediction mean : {ensemble_test.mean():.3f}')

# ============================================================
# 5-fold cross-validation (using optimized parameters), enable validation when True
# ============================================================
RUN_CV = False   
if RUN_CV:
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X_train))
    cv_rmses = []
    for fold, (tr_idx, vl_idx) in enumerate(kf.split(X_train)):
        print(f'--- Fold {fold+1}/5 ---')
        Xtr, Xvl = X_train[tr_idx], X_train[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        imp_cv = SimpleImputer(strategy='mean')
        Xtr = imp_cv.fit_transform(Xtr)
        Xvl = imp_cv.transform(Xvl)
        sc_cv = StandardScaler()
        Xtr = sc_cv.fit_transform(Xtr)
        Xvl = sc_cv.transform(Xvl)
        # Perform cross-validation with the best parameters found using Optuna
        m = XGBRegressor(**xgb_best_params)
        m.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], verbose=False)
        fold_pred = m.predict(Xvl)
        fold_rmse = np.sqrt(mean_squared_error(yvl, fold_pred))
        cv_rmses.append(fold_rmse)
        print(f'  Fold RMSE: {fold_rmse:.5f}')
    print(f'\nCV RMSE: {np.mean(cv_rmses):.5f} ± {np.std(cv_rmses):.5f}')
else:
    print('\nCV skipped (set RUN_CV = True to enable)')